# Run everything
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/run_all.ipynb)

Runs every stage in order (data → baselines → RQ1 → RQ2 → RQ3 → tiny Transformer → report).
Tick `SMOKE_TEST` for a ~10-minute end-to-end check on a handful of examples.
Full run: about 2–3 h on a T4 (less on L4/A100). Every stage resumes from what is already on Drive,
so after a disconnect just run the cells again.

In [ ]:
#@title Setup: clone the repo, install, choose where outputs go
#@markdown Outputs (data, adapters, predictions, metrics) persist on Google Drive so a
#@markdown disconnected runtime can resume: every stage skips work that is already saved.
USE_DRIVE = True  #@param {type:"boolean"}
SMOKE_TEST = False  #@param {type:"boolean"}
import os, sys
REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"
if SMOKE_TEST:
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = "--config configs/default.yaml" + (" --config configs/smoke.yaml" if SMOKE_TEST else "")
print("outputs ->", OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m cegqa.cli all $CONFIG

In [ ]:
from IPython.display import Markdown, display
import os
os.chdir(OUT)
display(Markdown(open("report.md").read()))
os.chdir(REPO_DIR)